# Test code

Check the Gurobi license

In [13]:
import gurobipy as gp
print(gp.gurobi.version())

(12, 0, 3)


Import of librabries

In [24]:
import json
from gurobipy import GRB
import networkx as nx
import time as time_setting

Import the input

In [3]:
instance = "01"
datasetPath = "../setA"

with open(f"{datasetPath}/setA-{instance}-net.json", "r") as netFile:
    net = json.load(netFile)

with open(f"{datasetPath}/setA-{instance}-scenario.json", "r") as scenarioFile:
    scenario = json.load(scenarioFile)

with open(f"{datasetPath}/setA-{instance}-tm.json", "r") as tmFile:
    tm = json.load(tmFile)

# Network file

V = [node["id"] for node in net["nodes"]]
A = [(link["from"], link["to"]) for link in net["links"]]

# Dict from arc (i, j) to capacity of arc (i, j)
c = {(link["from"], link["to"]): link["capacity"] for link in net["links"]}

# Dict from arc (i, j) to weight/metric of arc (i, j)
weight = {(link["from"], link["to"]): link["metric"] for link in net["links"]}

# Dict from link id to arc (i, j)
link_to_A = {link["id"]: (link["from"], link["to"]) for link in net["links"]}

# Dict from arc (i, j) to link id
A_to_link = {(link["from"], link["to"]): link["id"] for link in net["links"]}


# Scenario file

maxSeg = scenario["max_segments"]

# Dict from time t to budget value at time t
kappla = {i["t"]: i["value"] for i in scenario["budget"]}

# Dict from time t to list of link id (that needs an intervation at time t)
q = {i["t"]: i["links"] for i in scenario["interventions"]}

# Intervation expressed in a arc way instead of the id_link
# Dict from time t to list of arcs (i, j) (that needs an intervation at time t)
q_arc = {t: [link_to_A[link_id] for link_id in links] for t, links in q.items()}

# tm file

num_time_slots = tm["num_time_slots"]
T = list(range(num_time_slots))
T_star = T[1:]
D = list(range(len(tm["demands"])))  # demands indexed 0..n-1

# Dict from demand d to source node s(d)
s = {d: tm["demands"][d]["s"] for d in D}

# Dict from demand d to target node t(d)
t = {d: tm["demands"][d]["t"] for d in D}

# Dict from (d, t) to traffic volume of demand d at time time
v = {(d, time): tm["demands"][d]["v"][time] for d in D for time in T}

MIP model

In [15]:
# Offline computation

def buildNxGraph(net, V):
    G = nx.DiGraph()
    G.add_nodes_from(V)
    for link in net["links"]:
        G.add_edge(link["from"], link["to"], weight=link["metric"])
    return G

def computeSplitCoef(pairs, A, T, V, q_arc, net):
    G = buildNxGraph(net, V)
    r = {}
    
    for time in T:
        G_t = G.copy()
        G_t.remove_edges_from(q_arc.get(time, []))
        
        shortest_dist = dict(nx.all_pairs_dijkstra_path_length(G_t, weight="weight"))
        
        shortest_dist_reverse = dict(nx.all_pairs_dijkstra_path_length(G_t.reverse(), weight="weight"))
        
        for (i,j) in pairs:
            if j not in shortest_dist.get(i, {}):
                continue
            
            #d = shortest_dist[i][j]
            shortest_dist_i = shortest_dist[i] 
            FGList = []
            
            for (u,v) in G_t.edges():
                if i in shortest_dist and u in shortest_dist[i] and v in shortest_dist_reverse and j in shortest_dist[v]:
                    FGList.append((u,v))
            
            if not FGList:
                continue
            
            FG = nx.DiGraph()
            FG.add_nodes_from(V)
            FG.add_edges_from(FGList)
            
            # ECMP
            flow = {node: 0.0 for node in V}
            flow[i] = 1.0
            order = []
            for u in FG.nodes():
                if u in shortest_dist:
                    order.append((shortest_dist_i[u], u))
            order.sort()
            order = [u for _, u in order]
            for u in order:
                if u == j:
                    continue
                out = list(FG.out_edges(u))
                if not out:
                    continue
                split = flow[u] / len(out)
                for _, v in out:
                    r[(i, j, u, v, time)] = r.get((i, j, u, v, time), 0.0) + split
                    flow[v] += split
    return r
                    
# Offline computation

pairs = [(i, j) for i in V for j in V if i != j]

r = computeSplitCoef(pairs, A, T, V, q_arc, net)

In [5]:
m = gp.Model()

# Variable definition

# Binary variable: 1 if the path for demand d uses segment (i,j) at time t
x = m.addVars(D, T, V, V, vtype=GRB.BINARY, name="x")

# Load variable of arc a to time t
lambdaVar = m.addVars(A, T, lb=0.0, name="lambda")

# Buffer variable for the absolute value
y = m.addVars(D, T_star, V, V, lb=0.0, name="y")

# Constraint definition

# (a) Flow conservation constraints
for d in D:
    sd = s[d]
    td = t[d]
    for time in T:
        for i in V:
            buffer = 0
            if i == sd:
                buffer = 1
            elif i == td:
                buffer = -1
            m.addConstr((gp.quicksum(x[d, time, i, j] for j in V if i != j))-gp.quicksum(x[d, time, j, i] for j in V if i != j) == buffer)        
    

# (b) Number of segments
m.addConstrs(gp.quicksum(x[d, time, i, j] for (i, j) in pairs) <= maxSeg for d in D for time in T)

# (c) Load constraints

for time in T:
    ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
    for a in ANotInQ:
        m.addConstr(
            gp.quicksum(r.get((i, j, a[0], a[1], time),0.0) * v[d, time] * x[d, time, i, j] for d in D for (i, j) in pairs)
            <= lambdaVar[a[0], a[1], time] * c[a])

# (d) Budget constraints
m.addConstrs(gp.quicksum(y[d, time, i, j] for d in D for (i,j) in pairs) <= kappla[time] for time in T_star)
m.addConstrs(y[d, time, i, j] >= x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)
m.addConstrs(y[d, time, i, j] >= -x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)

# Objective
m.setObjective(0, GRB.MINIMIZE)
m.optimize()

if m.Status == GRB.OPTIMAL:
    print("Feasible solution")
else:
    print(f"No solution : {m.Status}")

Set parameter Username
Set parameter LicenseID to value 2712837
Academic license - for non-commercial use only - expires 2026-09-24
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 32240 rows, 48160 columns and 2428135 nonzeros
Model fingerprint: 0x63cd9719
Variable types: 16160 continuous, 32000 integer (32000 binary)
Coefficient statistics:
  Matrix range     [9e-05, 1e+03]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+01]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.09 seconds (0.03 work units)
Thread count was 1 (of 16 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00

Check of the found solution - Claude generated

In [6]:
# Extract the active segments (x variables)
print("Active segments (x[d, time, i, j] = 1):")
for d in D:
    for time in T:
        segments = [(i, j) for (i, j) in pairs if x[d, time, i, j].X > 0.5]
        print(f"  Demand {d} (s={s[d]}, t={t[d]}) at time {time}: {segments}")

Active segments (x[d, time, i, j] = 1):
  Demand 0 (s=7, t=8) at time 0: [(7, 8)]
  Demand 0 (s=7, t=8) at time 1: [(7, 8)]
  Demand 1 (s=15, t=0) at time 0: [(15, 0)]
  Demand 1 (s=15, t=0) at time 1: [(15, 0)]
  Demand 2 (s=2, t=3) at time 0: [(2, 3)]
  Demand 2 (s=2, t=3) at time 1: [(2, 3)]
  Demand 3 (s=16, t=12) at time 0: [(16, 12)]
  Demand 3 (s=16, t=12) at time 1: [(16, 12)]
  Demand 4 (s=9, t=6) at time 0: [(9, 6)]
  Demand 4 (s=9, t=6) at time 1: [(9, 6)]
  Demand 5 (s=8, t=17) at time 0: [(8, 17)]
  Demand 5 (s=8, t=17) at time 1: [(8, 17)]
  Demand 6 (s=18, t=14) at time 0: [(18, 14)]
  Demand 6 (s=18, t=14) at time 1: [(18, 14)]
  Demand 7 (s=17, t=0) at time 0: [(17, 0)]
  Demand 7 (s=17, t=0) at time 1: [(17, 0)]
  Demand 8 (s=0, t=18) at time 0: [(0, 18)]
  Demand 8 (s=0, t=18) at time 1: [(0, 18)]
  Demand 9 (s=18, t=3) at time 0: [(18, 3)]
  Demand 9 (s=18, t=3) at time 1: [(18, 3)]
  Demand 10 (s=9, t=14) at time 0: [(9, 14)]
  Demand 10 (s=9, t=14) at time 1: [(9,

In [7]:
# Check the load on each arc
print("\nArc loads λ(a, t):")
for time in T:
    for a in A:
        load = lambdaVar[a[0], a[1], time].X
        if load > 1e-6:  # only print non-zero loads
            print(f"  Arc {a} at time {time}: load = {load:.4f}  (capacity = {c[a]})")


Arc loads λ(a, t):
  Arc (0, 9) at time 0: load = 100000000.0000  (capacity = 250)
  Arc (9, 0) at time 0: load = 100000000.0000  (capacity = 877)
  Arc (1, 3) at time 0: load = 100000000.0000  (capacity = 513)
  Arc (3, 1) at time 0: load = 100000000.0000  (capacity = 383)
  Arc (2, 16) at time 0: load = 100000000.0000  (capacity = 981)
  Arc (16, 2) at time 0: load = 100000000.0000  (capacity = 391)
  Arc (3, 15) at time 0: load = 100000000.0000  (capacity = 486)
  Arc (15, 3) at time 0: load = 100000000.0000  (capacity = 435)
  Arc (4, 17) at time 0: load = 100000000.0000  (capacity = 591)
  Arc (17, 4) at time 0: load = 100000000.0000  (capacity = 469)
  Arc (5, 1) at time 0: load = 100000000.0000  (capacity = 277)
  Arc (1, 5) at time 0: load = 100000000.0000  (capacity = 817)
  Arc (6, 10) at time 0: load = 100000000.0000  (capacity = 667)
  Arc (10, 6) at time 0: load = 100000000.0000  (capacity = 489)
  Arc (7, 1) at time 0: load = 100000000.0000  (capacity = 685)
  Arc (1, 7)

The loads are very high since we fix neither an objective nor an upper bound on $\lambda$.

In [8]:
# Verify flow conservation manually
print("\nFlow conservation check:")
for d in D:
    for time in T:
        for i in V:
            out_flow = sum(x[d, time, i, j].X for j in V if i != j)
            in_flow  = sum(x[d, time, j, i].X for j in V if i != j)
            balance = out_flow - in_flow
            if abs(balance) > 1e-6:
                print(f"  d={d}, t={time}, node={i}: balance={balance:.4f}")


Flow conservation check:
  d=0, t=0, node=7: balance=1.0000
  d=0, t=0, node=8: balance=-1.0000
  d=0, t=1, node=7: balance=1.0000
  d=0, t=1, node=8: balance=-1.0000
  d=1, t=0, node=0: balance=-1.0000
  d=1, t=0, node=15: balance=1.0000
  d=1, t=1, node=0: balance=-1.0000
  d=1, t=1, node=15: balance=1.0000
  d=2, t=0, node=2: balance=1.0000
  d=2, t=0, node=3: balance=-1.0000
  d=2, t=1, node=2: balance=1.0000
  d=2, t=1, node=3: balance=-1.0000
  d=3, t=0, node=12: balance=-1.0000
  d=3, t=0, node=16: balance=1.0000
  d=3, t=1, node=12: balance=-1.0000
  d=3, t=1, node=16: balance=1.0000
  d=4, t=0, node=6: balance=-1.0000
  d=4, t=0, node=9: balance=1.0000
  d=4, t=1, node=6: balance=-1.0000
  d=4, t=1, node=9: balance=1.0000
  d=5, t=0, node=8: balance=1.0000
  d=5, t=0, node=17: balance=-1.0000
  d=5, t=1, node=8: balance=1.0000
  d=5, t=1, node=17: balance=-1.0000
  d=6, t=0, node=14: balance=-1.0000
  d=6, t=0, node=18: balance=1.0000
  d=6, t=1, node=14: balance=-1.0000
  d=

In [9]:
# Check the MLU
all_loads = [lambdaVar[a[0], a[1], time].X for a in A for time in T]
print(f"\nMax Link Utilization (MLU) = {max(all_loads):.4f}")


Max Link Utilization (MLU) = 100000000.0000


MLU is very high due to the fact the $\lambda$ are very high

Iteration all the instances in the setA :

In [37]:
datasetPath = "../setA"
for idx in range(1,21):
    t0 = time_setting.time()
    
    if idx < 10:
        instance = f"0{idx}"
    else:
        instance = f"{idx}"
    print("---------------------------------------------------------------------------------------------------------------------------")
    print(f"instance {instance}")
    print("---------------------------------------------------------------------------------------------------------------------------")
    with open(f"{datasetPath}/setA-{instance}-net.json", "r") as netFile:
        net = json.load(netFile)

    with open(f"{datasetPath}/setA-{instance}-scenario.json", "r") as scenarioFile:
        scenario = json.load(scenarioFile)

    with open(f"{datasetPath}/setA-{instance}-tm.json", "r") as tmFile:
        tm = json.load(tmFile)

    # Network file

    V = [node["id"] for node in net["nodes"]]
    A = [(link["from"], link["to"]) for link in net["links"]]

    # Dict from arc (i, j) to capacity of arc (i, j)
    c = {(link["from"], link["to"]): link["capacity"] for link in net["links"]}

    # Dict from arc (i, j) to weight/metric of arc (i, j)
    weight = {(link["from"], link["to"]): link["metric"] for link in net["links"]}

    # Dict from link id to arc (i, j)
    link_to_A = {link["id"]: (link["from"], link["to"]) for link in net["links"]}

    # Dict from arc (i, j) to link id
    A_to_link = {(link["from"], link["to"]): link["id"] for link in net["links"]}


    # Scenario file

    maxSeg = scenario["max_segments"]

    # Dict from time t to budget value at time t
    kappla = {i["t"]: i["value"] for i in scenario["budget"]}

    # Dict from time t to list of link id (that needs an intervation at time t)
    q = {i["t"]: i["links"] for i in scenario["interventions"]}

    # Intervation expressed in a arc way instead of the id_link
    # Dict from time t to list of arcs (i, j) (that needs an intervation at time t)
    q_arc = {t: [link_to_A[link_id] for link_id in links] for t, links in q.items()}

    # tm file

    num_time_slots = tm["num_time_slots"]
    T = list(range(num_time_slots))
    T_star = T[1:]
    D = list(range(len(tm["demands"])))  # demands indexed 0..n-1

    # Dict from demand d to source node s(d)
    s = {d: tm["demands"][d]["s"] for d in D}

    # Dict from demand d to target node t(d)
    t = {d: tm["demands"][d]["t"] for d in D}

    # Dict from (d, t) to traffic volume of demand d at time time
    v = {(d, time): tm["demands"][d]["v"][time] for d in D for time in T}
    
    # Offline computation
    
    pairs = [(i, j) for i in V for j in V if i != j]

    r = computeSplitCoef(pairs, A, T, V, q_arc, net)
    
    print(f"Offline computation: {time_setting.time()-t0} s")
    
    t1 = time_setting.time()
    
    m = gp.Model()

    m.Params.OutputFlag = 0 
    
    # Variable definition

    A_set = set(A) # To speed up the search
    
    # Binary variable: 1 if the path for demand d uses segment (i,j) at time t
    x = m.addVars(D, T, V, V, vtype=GRB.BINARY, name="x")

    # Load variable of arc a to time t
    lambdaVar = m.addVars(A_set, T, lb=0.0, name="lambda")

    # Buffer variable for the absolute value
    y = m.addVars(D, T_star, V, V, lb=0.0, name="y")

    print(f"Variable definition: {time_setting.time()-t1} s")
    
    t2 = time_setting.time()
    
    # Constraint definition

    # (a) Flow conservation constraints
    for d in D:
        sd = s[d]
        td = t[d]
        for time in T:
            for i in V:
                buffer = 0
                if i == sd:
                    buffer = 1
                elif i == td:
                    buffer = -1
                m.addConstr(gp.quicksum(x[d,time,i,j] for j in V if (i,j) in pairs)
                            - gp.quicksum(x[d,time,j,i] for j in V if (j,i) in pairs) == buffer)
    print(f"Flow conservation constraints: {time_setting.time()-t2} s")
    
    t3 = time_setting.time()
    
    # (b) Number of segments
    m.addConstrs(gp.quicksum(x[d, time, i, j] for (i, j) in pairs) <= maxSeg for d in D for time in T)

    print(f"Number of segments constraints: {time_setting.time()-t3} s")
    
    t4 = time_setting.time()
    
    # (c) Load constraints - speeded up
    
    nonzero_r = {}
    for (i, j, a0, a1, t), val in r.items():
        if val > 1e-9:
            if (i, j) not in A_set:
                continue
            key = ((a0, a1), t)
            if key not in nonzero_r:
                nonzero_r[key] = []
            nonzero_r[key].append((i, j, val))
    
    for time in T:
        ANotInQ = [(i, j) for (i, j) in A_set if (i, j) not in q_arc.get(time, [])]
        for a in ANotInQ:
            terms = nonzero_r.get((a, time), [])
            if not terms:
                continue
            m.addConstr(
                gp.quicksum(
                    r_val * v[d, time] * x[d, time, i, j]
                    for (i, j, r_val) in terms
                    for d in D
                ) <= lambdaVar[a[0], a[1], time] * c[a]
            )
    
    """
    # (c) Load constraints
    
    for time in T:
        ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
        for a in ANotInQ:
            m.addConstr(
                gp.quicksum(r.get((i, j, a[0], a[1], time),0.0) * v[d, time] * x[d, time, i, j] for d in D for (i, j) in pairs)
                <= lambdaVar[a[0], a[1], time] * c[a])
    """

    print(f"Load constraints: {time_setting.time()-t4} s")
    
    t5 = time_setting.time()

    # (d) Budget constraints
    m.addConstrs(gp.quicksum(y[d, time, i, j] for d in D for (i,j) in A) <= kappla[time] for time in T_star)
    m.addConstrs(y[d, time, i, j] >= x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)
    m.addConstrs(y[d, time, i, j] >= -x[d, time, i, j]+ x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)

    print(f"Budget constraints: {time_setting.time()-t5} s")
    
    t6 = time_setting.time()
    
    # Objective
    m.setObjective(0, GRB.MINIMIZE)
    m.optimize()

    print(f"Objective and optimization step: {time_setting.time()-t6} s")
    print(f"Total time for the instance {idx} : {time_setting.time()-t0} s")
    
    if m.Status == GRB.OPTIMAL:
        print("Feasible solution")
    else:
        print(f"No solution : {m.Status}")

---------------------------------------------------------------------------------------------------------------------------
instance 01
---------------------------------------------------------------------------------------------------------------------------
Offline computation: 0.20287108421325684 s
Variable definition: 0.6286482810974121 s
Flow conservation constraints: 0.263211727142334 s
Number of segments constraints: 0.02107834815979004 s
Load constraints: 0.7004039287567139 s
Budget constraints: 0.6116068363189697 s
Objective and optimization step: 0.045368194580078125 s
Total time for the instance 1 : 2.4746367931365967 s
Feasible solution
---------------------------------------------------------------------------------------------------------------------------
instance 02
---------------------------------------------------------------------------------------------------------------------------
Offline computation: 0.6260097026824951 s
Variable definition: 0.4512460231781006 s

KeyboardInterrupt: 

Algorithms and heuristics

In [ ]:
# Objective
m.update()
zlist = []
for a in A:
    for time in T:
        z = m.addVar(lb=0.0)
        tempConstrs = m.addConstrs(lambdaVar[a[0], a[1], time] <= z for a in A for time in T)
        m.setObjective(z, GRB.MINIMIZE)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            break
        zlist.append(z.X)
        m.addConstrs(lambdaVar[a[0], a[1], time] <= z.X for a in A for time in T)
        m.remove(list(tempConstrs.values()))
        m.remove(z)
        if len(zlist) > 1 and abs(zlist[-1] - zlist[-2]) < 1e-6:
            break

Results and performance analysis